# Neural Networks

## What is a Neural Network?
- A **neural network** is a collection of interconnected nodes (neurons) organized in layers.
- Each neuron computes a weighted sum of its inputs plus a bias, then applies a nonlinear activation.
- Together, neurons approximate complex functions by learning weights during training.

___

## Layers of a Neural Network
![Neural Network Structure](Colored_neural_network.png)
- **Input layer:** Accepts raw features, no parameters.
- **Hidden layer(s):** Transform inputs via weights, biases, and activations.
- **Output layer:** Computes final predictions (e.g., class scores, regression values).

Deep networks have multiple hidden layers for rich feature hierarchies.

___

## Forward Pass & Activation
For each neuron:
1. Compute weighted sum: $$z = W x + b$$
2. Apply activation: $$a = \sigma(z)$$

**Forward pass:** propagate inputs layer by layer to get outputs.

___

## Activation Functions
Activation introduces nonlinearity, enabling networks to learn complex patterns.

- **Sigmoid:** $$\sigma(z)=\frac{1}{1+e^{-z}}$$
- **Tanh:** $$\tanh(z)=\frac{e^{z}-e^{-z}}{e^{z}+e^{-z}}$$
- **ReLU:** $$\mathrm{ReLU}(z)=\max(0, z)$$

![Activation Functions](ActivationFunctions.png)

___

## Loss Functions
Quantify prediction error; training minimizes loss.

- **Mean Squared Error (MSE):** $$\frac{1}{N}\sum_{i}(y_i - \hat y_i)^2$$
- **Cross-Entropy:** $$-\frac{1}{N}\sum_i [y_i\log p_i + (1-y_i)\log(1-p_i)]$$

Choose loss based on the task (regression vs. classification).

___

## Training via Gradient Descent
Adjust weights to minimize loss. Steps:
1. Compute loss: $$L(\theta)$$
2. Compute gradient: $$\nabla_\theta L$$
3. Update weights: $$\theta := \theta - \alpha \nabla_\theta L$$

![Gradient Descent](GradientDescent.png)

Repeat until convergence.

___

## Backpropagation Algorithm
- Efficiently computes gradients of loss wrt each weight using the chain rule.
- For a weight w affecting output a via z: $$a=\sigma(z), z=wx+b$$
  $$\frac{\partial L}{\partial w}=\frac{\partial L}{\partial a}\frac{\partial a}{\partial z}\frac{\partial z}{\partial w}$$
- Propagate error derivatives backward through layers to update all weights.

___

## Training Loop Pseudocode
```julia
model = initialize_network()
for epoch in 1:NumEpochs
  for (x, y) in training_data
    # Forward
    ŷ = model(x)
    # Loss
    l = loss(ŷ, y)
    # Backward
    grads = gradient(() -> loss(model(x), y), params(model))
    # Update
    update!(optimizer, params(model), grads)
  end
end
``` 

Iterate epochs to improve performance.

___

## Example
* Go back to Neoclassical Model
\begin{align*}
    C_t^{-\gamma} &= \beta\mathbb E_t[R_{t+1}C_{t+1}^{-\gamma}]\\
    R_t &= 1 - \delta + \alpha A_t K_t^{\alpha -1}\\
    C_t + K_{t+1} &= A_t K_t^\alpha + (1-\delta)K_t\\
    A_t &= \rho A_{t-1} + \sigma \varepsilon_{t+1}
\end{align*}
* How to solve this with Neural Networks?

## Setup

In [ ]:
### Julia environment setup
using Flux,Statistics,Random,Parameters,Plots,DataFrames
# Set random seed for reproducibility
Random.seed!(42)
### Model parameters for the RBC model (all Float32 for GPU compatibility)
@with_kw mutable struct Params
    α::Float32 = 0.30      # capital share in production
    β::Float32 = 0.95      # discount factor
    δ::Float32 = 0.1       # depreciation rate
    γ::Float32 = 2.0       # CRRA utility coefficient (risk aversion)
    ρ::Float32 = 0.90      # persistence of productivity shock
    σ_ε::Float32 = 0.02      # std dev of shock innovation (for log A) 


    #bounds
    k_bounds::Tuple{Float32,Float32} = (0.5f0,1.5f0) #fraction of SS capital
    A_bounds::Tuple{Float32,Float32} = (0.5f0,1.5f0)
    β_bounds::Tuple{Float32,Float32} = (0.9f0,0.99f0)
    ρ_bounds::Tuple{Float32,Float32} = (0.9f0,0.99f0)
    γ_bounds::Tuple{Float32,Float32} = (0.5f0,4.0f0)
end

# marginal utility function for CRRA utility
function u_prime(c,γ)
    return c.^(-γ)      # derivative of c^(1-γ)/(1-γ) is c^(-γ)
end

## Steady State
* It's always helpful to know about where the ergodic distribution will be

In [ ]:
# Steady-state (for A=1) for reference (solve α β (A) k^(α-1) + β(1-δ) = 1)
function steady_state(α, β, δ)   
    A_ss = 1.0f0
    k_ss = @. ((1f0/β - (1f0-δ)) / (α * A_ss))^(1f0/(α-1f0))
    y_ss = @. A_ss * k_ss^α
    c_ss = @. y_ss - δ * k_ss

    return k_ss, c_ss, y_ss, A_ss
end

para = Params()
k_ss, c_ss, y_ss, A_ss = steady_state(para.α, para.β, para.δ)
println("Steady-state capital (A=1) ≈ ", round.(k_ss,digits=3))


## Normalizing Variables
* Ideally want to normalize variables to be on same scale [0,1]
* Helps network learn efficienty

In [ ]:
#Normalize variables
function normalize(x,x_low,x_high)
    return (x .- x_low) ./ (x_high - x_low)
end
function denormalize(x,x_low,x_high)
    return x .* (x_high - x_low) .+ x_low
end

## Defining the model
* Key objects
    * number of inputs (capital,TFP)
    * number of neurons per hidden layer
    * numer of hidden layers
    * number of outputs

In [ ]:
#Now construct the neural network 
model = Chain(
    Dense(2, 16, relu),
    Dense(16, 16, relu),
    Dense(16, 1, softplus)  # output layer (smooth version of RELU)
)

## Creating Sample Data
* Each epoch will sample from possible state variables
* Draw normalized sample

In [ ]:
"""
    sample_batch(para::Params, batch_size)

Generates sample of state variables
"""
function sample_batch(para::Params, batch_size)
    @unpack σ_ε, ρ, k_bounds, A_bounds, β_bounds = para
    # We sample uniform in log(k) to ensure lower end has coverage (since distribution of k might be skewed)
    k_batch = rand(Float32,batch_size)
    # sample productivity A (log-normal around 1)
    # We approximate the ergodic distribution of A_t as lognormal with mean 0 and std derived from σ_ε.
    # To be precise, if A follows log AR(1), its stationary distribution is N(0, σ_stat^2) with σ_stat^2 = σ_ε^2 / (1-ρ^2).
    σ_stat = σ_ε ./ sqrt.(1f0 .- ρ.^2f0)
    A = exp.(σ_stat .* randn(Float32,batch_size))
    A_batch = normalize(A,A_bounds[1],A_bounds[2])

    return Float32.(k_batch), Float32.(A_batch)
end

## Create Loss Function

* Start with batch of state data

* Evaluate policy given current neural network

* Draw productivity shock for the next period

* Evaluate EE errors as loss function

In [ ]:
function euler_residual_batch(para,model, data)
    @unpack α, β, δ, γ, σ_ε, ρ, k_bounds, A_bounds= para
    k_batch, A_batch = data
    k_ss, c_ss, y_ss, A_ss = steady_state(α, β, δ)
    batch_size = length(k_batch)
    k_low = k_bounds[1]*k_ss
    k_high = k_bounds[2]*k_ss
    
    X = hcat(k_batch, A_batch)'  # shape 2 x batch_size
    # model(X) will give a 1 x N matrix (or vector) of outputs
    model_out = model(X)            # 1 x batch_size

    #output fraction of resources consumed
    frac = vec(model_out)  # reshape to 1D vector length = batch_size
    
    #denormalize variables
    k = denormalize(k_batch,k_low,k_high)
    A = denormalize(A_batch,A_bounds[1],A_bounds[2])
    # Use broadcasting to compute vectorized consumption given states
    c = frac .* (A .* (k .^ α) .+ (1f0 .- δ).*k)

    # Now compute Euler residual for each element
    # u'(c) = c^(-γ)
    #marginal_utility = c .^ (-γ)
    # Next state capital k' for each state:
    k_next = (1f0 .- δ).*k .+ A .* (k .^ α) .- c
    k_next = clamp.(k_next, k_low, k_high) #NECESSARY FOR CONVERGENCE
    c  = (1f0 .- δ).*k .+ A .* (k .^ α) .- k_next
    marginal_utility = c .^ (-γ)
 
    # Draw one sample for next shock A' (for each current state) to approximate expectation
    # We draw A' given current A: log A' = ρ log A + σε * ε. We sample ε ~ N(0,1).
    eps = randn(Float32,batch_size)
    A_next = exp.(log.(A).*ρ .+ σ_ε .* eps)
    #normalize to feed into network
    A_next_norm = normalize(A_next,A_bounds[1],A_bounds[2])
    k_next_norm = normalize(k_next,k_low,k_high)
    X_next = hcat(k_next_norm, A_next_norm)'
    frac_next = vec(model(X_next))

    resource_next = A_next .* (k_next .^ α) .+ (1f0 .- δ).*k_next
    k_next_next = (1f0 .-frac_next).*resource_next
    k_next_next = clamp.(k_next_next, k_low, k_high)
    c_next = resource_next - k_next_next
    # marginal utility next period
    marginal_utility_next = c_next .^ (-γ)

    # Compute RHS of Euler: β * u'(c_{t+1}) * (α A' k'^{α-1} + 1-δ)
    # Calculate return factor (gross) = α A' k'^{α-1} + 1 - δ
    R_next = α .* A_next .* (k_next .^ (α .- 1f0)) .+ (1f0 .- δ)
    rhs = β .* marginal_utility_next .* R_next

    # Residual = LHS - RHS
    residuals = rhs .- marginal_utility
    return residuals  # vector of length batch_size
end

In [ ]:
# Define loss function
function loss_fn(model,data)
    residuals = euler_residual_batch(para, model, data)
    return mean(residuals .^ 2f0)
end

## Setup Learning

In [ ]:
# Hyperparameters
batch_size = 2048      # number of state points per batch
epochs     = 10000      # training epochs
η          = 1e-3      # initial learning rate

# Initialize optimizer
opt_state = Flux.setup(Adam(η), model) #ADAM is an efficient optimizer

## Do Training

In [ ]:

# Training loop
println("Starting training...")
losses = []
for epoch in 1:epochs
    # Sample a batch of states
    data = sample_batch(para, batch_size)
    # Compute gradient of loss (mean squared residual) w.r.t. model parameters
    val, grads = Flux.withgradient(model) do m
        loss_fn(m, data)
    end
    Flux.update!(opt_state, model, grads[1])
    # (Optional) decay learning rate or print progress
    push!(losses, val)
    if epoch % 100 == 0
        # Compute current loss for reporting
        println("Epoch $epoch, training MSE Euler residual = $(round(val, sigdigits=3))")
    end
end

## Learning Path

In [ ]:
plot(losses[200:end], title="Training Loss", xlabel="Epochs", ylabel="MSE Euler Residual", legend=false)

## Simulation

In [ ]:

# Function to simulate the economy using the trained model
function simulate_economy(model, para::Params, T::Int=200, k0=nothing, A0=nothing)
    @unpack α, β, δ, γ, ρ, σ_ε, k_bounds, A_bounds = para
    k_ss, c_ss, y_ss, A_ss = steady_state(α, β, δ)
    k_low = k_bounds[1]*k_ss
    k_high = k_bounds[2]*k_ss
    # If initial values not provided, use steady state values
    if isnothing(k0)
        k0 = k_ss
    end
    if isnothing(A0)
        A0 = A_ss
    end
    
    # Initialize arrays to store time series
    k_series = zeros(Float32, T+1)
    A_series = zeros(Float32, T+1)
    c_series = zeros(Float32, T)
    y_series = zeros(Float32, T)
    i_series = zeros(Float32, T)
    
    # Set initial values
    k_series[1] = k0
    A_series[1] = A0
    
    # Generate all random shocks in advance
    ε_series = randn(Float32, T)
    
    # Simulation loop
    for t in 1:T
        # Current state
        k = k_series[t]
        A = A_series[t]
        
        # Normalize state for model input
        k_norm = normalize(k, k_low, k_high)
        A_norm = normalize(A, A_bounds[1], A_bounds[2])
        
        # Create input for neural network
        state = reshape([k_norm, A_norm], 2, 1)
        
        # Get consumption fraction from model
        frac = model(state)[1]
        
        # Calculate current output
        y = A * k^α
        y_series[t] = y
        
        # Calculate consumption
        resources = y + (1f0 - δ) * k
        c = frac * resources
        c_series[t] = c
        
        # Calculate investment
        i = y - c
        i_series[t] = i
        
        # Next-period capital - FIX: proper capital accumulation equation
        k_next = (1f0 - δ) * k + i  # This was incorrect before (was just i)
        k_series[t+1] = max(k_next, 1e-6)  # Prevent negative capital
        
        # Next-period productivity (AR(1) process)
        logA_next = ρ * log(A) + σ_ε * ε_series[t]
        A_series[t+1] = exp(logA_next)
    end
    
    # Create DataFrame with results
    results = DataFrame(
        capital = k_series[1:T],
        productivity = A_series[1:T],
        consumption = c_series,
        output = y_series,
        investment = i_series,
        savings_rate = i_series ./ y_series
    )
    
    # Add metadata as properties
    results.k_ss = k_ss*ones(T)
    results.c_ss = c_ss*ones(T)
    results.y_ss = y_ss*ones(T)
    results.A_ss = A_ss*ones(T)
    results.β = β*ones(T)
    results.ρ = ρ*ones(T)
    
    return results
end


## Construct IRF

In [ ]:
# Function to compute impulse response to productivity shock
function impulse_response(model, para::Params, shock_size=0.05, T=40,NIRF=20)
    k_ss, c_ss, y_ss, A_ss = steady_state(para.α, para.β, para.δ)
    # Run two simulations: one baseline and one with a shock
    # Initial conditions: steady state
    Random.seed!(96)
    baseline = simulate_economy(model, para, T, k_ss, A_ss)
    for i in 1:NIRF-1
        baseline = baseline .+ simulate_economy(model, para, T, k_ss, A_ss)
    end
    baseline = baseline ./ NIRF
    # Shocked economy: shock_size% higher productivity initially
    Random.seed!(96)
    shocked = simulate_economy(model, para, T, k_ss, A_ss * (1 + shock_size))
    for i in 1:NIRF-1
        shocked = shocked .+ simulate_economy(model, para, T, k_ss, A_ss * (1 + shock_size))
    end
    shocked = shocked ./ NIRF
    
    # Calculate percentage deviations from baseline
    deviations = DataFrame()
    
    # Calculate deviations for each variable
    deviations.output = 100 * (shocked.output ./ baseline.output .- 1)
    deviations.consumption = 100 * (shocked.consumption ./ baseline.consumption .- 1)
    deviations.investment = 100 * (shocked.investment ./ baseline.investment .- 1)
    deviations.capital = 100 * (shocked.capital ./ baseline.capital .- 1)
    
    # Add metadata
    deviations.β = baseline.β
    deviations.ρ = baseline.ρ
    
    # Plot impulse responses
    p = plot(title="Impulse Responses to $(shock_size*100)% Productivity Shock (β=$(round(baseline.β[1],digits=3)), ρ=$(round(baseline.ρ[1],digits=3)))", 
             xlabel="Time", ylabel="% Deviation from Baseline",
             legend=:topright, size=(800, 500))
    
    plot!(p, 0:T-1, deviations.output, label="Output", linewidth=2)
    plot!(p, 0:T-1, deviations.consumption, label="Consumption", linewidth=2)
    plot!(p, 0:T-1, deviations.investment, label="Investment", linewidth=2)
    plot!(p, 0:T-1, deviations.capital, label="Capital", linewidth=2)
    
    return p, deviations
end

## Plotting Results

In [ ]:
# Generate impulse response
println("\nComputing impulse response to productivity shock...")
ir_plot, deviations = impulse_response(model, para, 0.05, 40, 50)
display(ir_plot)

## Pros and Cons 
* Flexible: only wrote down code to simulate and check EE erros

* Training time: took longer to train the model than solving with interpolation

* Can we improve this?
    - maybe solve larger model over set of parameters
___

## Updated Creating Sample

In [ ]:
"""
    sample_batch(para::Params, batch_size)

Generates sample of state variables
"""
function sample_batch(para::Params, batch_size)
    @unpack σ_ε, ρ, k_bounds, A_bounds, β_bounds = para
    # We sample uniform in log(k) to ensure lower end has coverage (since distribution of k might be skewed)
    k_batch = rand(Float32,batch_size)
    β_batch = rand(Float32,batch_size)
    ρ_batch = rand(Float32,batch_size)
    γ_batch = rand(Float32,batch_size)
    # sample productivity A (log-normal around 1)
    # We approximate the ergodic distribution of A_t as lognormal with mean 0 and std derived from σ_ε.
    # To be precise, if A follows log AR(1), its stationary distribution is N(0, σ_stat^2) with σ_stat^2 = σ_ε^2 / (1-ρ^2).
    σ_stat = σ_ε ./ sqrt.(1f0 .- ρ.^2f0)
    A = exp.(σ_stat .* randn(Float32,batch_size))
    A_batch = normalize(A,A_bounds[1],A_bounds[2])

    return Float32.(k_batch), Float32.(A_batch), Float32.(β_batch), Float32.(ρ_batch), Float32.(γ_batch)
end

## Updated Construct EE residuals

In [ ]:
function euler_residual_batch(para,model, data)
    @unpack α, β, δ, γ, σ_ε, ρ, k_bounds, A_bounds,β_bounds,ρ_bounds,γ_bounds= para
    k_batch, A_batch, β_batch, ρ_batch, γ_batch = data
    β = denormalize(β_batch,β_bounds[1],β_bounds[2])
    ρ = denormalize(ρ_batch,ρ_bounds[1],ρ_bounds[2])
    γ = denormalize(γ_batch,γ_bounds[1],γ_bounds[2])
    k_ss, c_ss, y_ss, A_ss = steady_state(α, β, δ)
    batch_size = length(k_batch)
    k_low = k_bounds[1]*k_ss
    k_high = k_bounds[2]*k_ss
    
    X = hcat(k_batch, A_batch, β_batch, ρ_batch, γ_batch)' # shape 5 x batch_size
    # model(X) will give a 1 x N matrix (or vector) of outputs
    model_out = model(X)            # 1 x batch_size

    #output fraction of resources consumed
    frac = vec(model_out)  # reshape to 1D vector length = batch_size
    
    #denormalize variables
    k = denormalize(k_batch,k_low,k_high)
    A = denormalize(A_batch,A_bounds[1],A_bounds[2])
    # Use broadcasting to compute vectorized consumption given states
    c = frac .* (A .* (k .^ α) .+ (1f0 .- δ).*k)

    # Now compute Euler residual for each element
    # u'(c) = c^(-γ)
    #marginal_utility = c .^ (-γ)
    # Next state capital k' for each state:
    k_next = (1f0 .- δ).*k .+ A .* (k .^ α) .- c
    k_next = clamp.(k_next, k_low, k_high) #NECESSARY FOR CONVERGENCE
    c  = (1f0 .- δ).*k .+ A .* (k .^ α) .- k_next
    marginal_utility = c .^ (-γ)
 
    # Draw one sample for next shock A' (for each current state) to approximate expectation
    # We draw A' given current A: log A' = ρ log A + σε * ε. We sample ε ~ N(0,1).
    eps = randn(Float32,batch_size)
    A_next = exp.(log.(A).*ρ .+ σ_ε .* eps)
    #normalize to feed into network
    A_next_norm = normalize(A_next,A_bounds[1],A_bounds[2])
    k_next_norm = normalize(k_next,k_low,k_high)
    X_next = hcat(k_next_norm, A_next_norm, β_batch, ρ_batch, γ_batch)'
    frac_next = vec(model(X_next))

    resource_next = A_next .* (k_next .^ α) .+ (1f0 .- δ).*k_next
    k_next_next = (1f0 .-frac_next).*resource_next
    k_next_next = clamp.(k_next_next, k_low, k_high)
    c_next = resource_next - k_next_next
    # marginal utility next period
    marginal_utility_next = c_next .^ (-γ)

    # Compute RHS of Euler: β * u'(c_{t+1}) * (α A' k'^{α-1} + 1-δ)
    # Calculate return factor (gross) = α A' k'^{α-1} + 1 - δ
    R_next = α .* A_next .* (k_next .^ (α .- 1f0)) .+ (1f0 .- δ)
    rhs = β .* marginal_utility_next .* R_next

    # Residual = LHS - RHS
    residuals = rhs .- marginal_utility
    return residuals  # vector of length batch_size
end

## Update Simulate Once Model Trained

In [ ]:
# Function to simulate the economy using the trained model
function simulate_economy(model, para::Params, T::Int=200, k0=nothing, A0=nothing)
    @unpack α, β, δ, γ, ρ, σ_ε, k_bounds, A_bounds,β_bounds,ρ_bounds,γ_bounds = para
    k_ss, c_ss, y_ss, A_ss = steady_state(α, β, δ)
    k_low = k_bounds[1]*k_ss
    k_high = k_bounds[2]*k_ss
    # If initial values not provided, use steady state values
    if isnothing(k0)
        k0 = k_ss
    end
    if isnothing(A0)
        A0 = A_ss
    end
    
    # Initialize arrays to store time series
    k_series = zeros(Float32, T+1)
    A_series = zeros(Float32, T+1)
    c_series = zeros(Float32, T)
    y_series = zeros(Float32, T)
    i_series = zeros(Float32, T)
    
    # Set initial values
    k_series[1] = k0
    A_series[1] = A0
    
    # Generate all random shocks in advance
    ε_series = randn(Float32, T)
    
    # Simulation loop
    for t in 1:T
        # Current state
        k = k_series[t]
        A = A_series[t]
        
        # Normalize state for model input
        k_norm = normalize(k, k_low, k_high)
        A_norm = normalize(A, A_bounds[1], A_bounds[2])
        β_norm = normalize(β, β_bounds[1], β_bounds[2])
        ρ_norm = normalize(ρ, ρ_bounds[1], ρ_bounds[2])
        γ_norm = normalize(γ, γ_bounds[1], γ_bounds[2])
        
        # Create input for neural network
        state = reshape([k_norm, A_norm, β_norm, ρ_norm, γ_norm], 5, 1)
        
        # Get consumption fraction from model
        frac = model(state)[1]
        
        # Calculate current output
        y = A * k^α
        y_series[t] = y
        
        # Calculate consumption
        resources = y + (1f0 - δ) * k
        c = frac * resources
        c_series[t] = c
        
        # Calculate investment
        i = y - c
        i_series[t] = i
        
        # Next-period capital - FIX: proper capital accumulation equation
        k_next = (1f0 - δ) * k + i  # This was incorrect before (was just i)
        k_series[t+1] = max(k_next, 1e-6)  # Prevent negative capital
        
        # Next-period productivity (AR(1) process)
        logA_next = ρ * log(A) + σ_ε * ε_series[t]
        A_series[t+1] = exp(logA_next)
    end
    
    # Create DataFrame with results
    results = DataFrame(
        capital = k_series[1:T],
        productivity = A_series[1:T],
        consumption = c_series,
        output = y_series,
        investment = i_series,
        savings_rate = i_series ./ y_series
    )
    
    # Add metadata as properties
    results.k_ss = k_ss*ones(T)
    results.c_ss = c_ss*ones(T)
    results.y_ss = y_ss*ones(T)
    results.A_ss = A_ss*ones(T)
    results.β = β*ones(T)
    results.ρ = ρ*ones(T)
    
    return results
end

## Update NN

In [ ]:
#Now construct the neural network 
model = Chain(
    Dense(5, 64, relu),
    Dense(64, 64, relu),
    Dense(64, 1, softplus)  # output layer (smooth version of RELU)
)

## Train NN

In [ ]:
# Initialize optimizer
opt_state = Flux.setup(Adam(η), model) #ADAM is an efficient optimizer
# Training loop
println("Starting training...")
losses = []
for epoch in 1:epochs
    # Sample a batch of states
    data = sample_batch(para, batch_size)
    # Compute gradient of loss (mean squared residual) w.r.t. model parameters
    val, grads = Flux.withgradient(model) do m
        loss_fn(m, data)
    end
    Flux.update!(opt_state, model, grads[1])
    # (Optional) decay learning rate or print progress
    push!(losses, val)
    if epoch % 100 == 0
        # Compute current loss for reporting
        println("Epoch $epoch, training MSE Euler residual = $(round(val, sigdigits=3))")
    end
end

## Plot IRFs

In [ ]:
# Generate impulse response
println("\nComputing impulse response to productivity shock...")
ir_plot, deviations = impulse_response(model, para, 0.05, 40, 50)
display(ir_plot)

In [ ]:
# Generate impulse response
println("\nComputing impulse response to productivity shock...")
para.β = 0.9f0
ir_plot, deviations = impulse_response(model, para, 0.05, 40, 50)
display(ir_plot)